In [29]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

___
Testing out the transcript library

In [30]:
video_id = 'syE8f_ow0iM'
ytt_api = YouTubeTranscriptApi()

In [31]:
print(ytt_api.list(video_id=video_id))

For this video (syE8f_ow0iM) transcripts are available in the following languages:

(MANUALLY CREATED)
None

(GENERATED)
 - en ("English (auto-generated)")[TRANSLATABLE]
 - fr-FR ("French (France) (auto-generated)")[TRANSLATABLE]
 - de-DE ("German (Germany) (auto-generated)")[TRANSLATABLE]
 - hi ("Hindi (auto-generated)")[TRANSLATABLE]
 - id ("Indonesian (auto-generated)")[TRANSLATABLE]
 - it ("Italian (auto-generated)")[TRANSLATABLE]
 - pt-BR ("Portuguese (Brazil) (auto-generated)")[TRANSLATABLE]
 - es-US ("Spanish (United States) (auto-generated)")[TRANSLATABLE]

(TRANSLATION LANGUAGES)
 - ar ("Arabic")
 - zh-Hant ("Chinese (Traditional)")
 - nl ("Dutch")
 - en ("English")
 - fr ("French")
 - de ("German")
 - hi ("Hindi")
 - id ("Indonesian")
 - it ("Italian")
 - ja ("Japanese")
 - ko ("Korean")
 - pt ("Portuguese")
 - ru ("Russian")
 - es ("Spanish")
 - th ("Thai")
 - uk ("Ukrainian")
 - vi ("Vietnamese")


___
Trying to obtain transcript for a video

In [32]:
try:
    transcript_list = ytt_api.fetch(video_id=video_id)
    print('Length of list: ', len(transcript_list), end='\n\n')
    print(transcript_list[0], end='\n\n')
    print('Type of each transcript snippet item: ', type(transcript_list[0]))

except TranscriptsDisabled:
    print('No transcript for this video')

Length of list:  2404

FetchedTranscriptSnippet(text='What is up ladies and gentlemen? Welcome', start=0.24, duration=4.32)

Type of each transcript snippet item:  <class 'youtube_transcript_api._transcripts.FetchedTranscriptSnippet'>


In [33]:
transcript = ' '.join(chunk.text for chunk in transcript_list).replace('\n', ' ')
print('Joined transcript total length:\n', len(transcript))

Joined transcript total length:
 86691


Transcript obtained successfully 

___
Splitting transcript into smaller chunks. Using RecursiveTextSplitter for this job

In [34]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

In [35]:
chunks = splitter.create_documents([transcript])
print('No of chunks: ', len(chunks), end='\n\n')
print('First chunk Document:\n', chunks[0], end='\n\n')

No of chunks:  134

First chunk Document:
 page_content='What is up ladies and gentlemen? Welcome back to Fudge Muppet. I'm Scott here with Michael and today we're talking all the law you need to know for Elder Scrolls 6. >> Yes. And I have a feeling this is going to be a series. Yeah. Series that people have been wanting for a long time. Dare I say the return of the Elder Scrolls 6 podcast. Maybe potentially probably. Now, we've been waiting for Elder Scrolls 6 for a very, very, very long time. In fact, since high school. We made this channel when we were wearing school uniforms and we would joke about how we'll have kids playing The Elder Scrolls 6. Well, I can tell you what, we need to wear glasses for this episode. >> We're not going to be able to read >> cringe glasses reveal, but we literally need to wear glasses. We have gray hairs on the'



Created chunks successfully

___
Embedding and storing into vectorstore

In [36]:
print(type(chunks[0]))
print(type(chunks))

<class 'langchain_core.documents.base.Document'>
<class 'list'>


In [37]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

embedder = OpenAIEmbeddings(
    model='local bge-m3',
    base_url='http://127.0.0.1:8081/v1',
    api_key='not needed' # type: ignore
)


chroma_store = Chroma(
    # persist_directory='Notebook testing DB',
    collection_name='notebook_test_collection',
    embedding_function=embedder
)

chroma_store.delete_collection()

chroma_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedder,
    # persist_directory='Notebook testing DB',
    collection_name='notebook_test_collection'
)


In [38]:
chroma_store.get(include=[])

{'ids': ['cda24771-0f14-4533-bcd6-1392d2774515',
  '20310cad-26f6-49d5-bb7e-aba93cd53275',
  'a95b1dab-976e-44bc-99ab-29183200e101',
  '9db3ff52-ec99-4fbf-9aa5-b752e0361eed',
  'd0ce3046-1f55-443f-a99c-5fab828b216d',
  'd4ad7408-d493-4be7-afea-185edb4cacb0',
  '24cd5f03-442a-4ae4-8043-e3ec689aeff7',
  '97875a41-43c3-4b50-8078-0750e7c1af09',
  'b4ec6488-327c-4e0d-953a-354273ee46d8',
  '3dfacc41-0848-4a5a-ba5f-da020d90b0aa',
  'c7e7768a-0611-4e14-942d-5ab7abbeb5f1',
  '87e0b312-738b-4b72-a37a-fdce67af8449',
  '27c24bf4-2253-4cab-8d9c-aed03d663be0',
  '9830b9ae-678a-4af3-b3dd-956c02ca15cc',
  'ee3ac0ef-7669-46dc-ba55-8b31e2ac93aa',
  '9e6ae432-fd57-41ab-aca8-c65d0a27de7a',
  '500099ae-2a54-4a46-9682-575d75696250',
  '0fd7571c-7cb2-41f3-8f29-33960067957a',
  'f3034d02-ad3f-4433-b786-dc99dd4b2363',
  'f674f824-19d7-493d-8744-3bec789e0e9a',
  '18b65527-d7d7-4682-8208-4ff163cdc52e',
  '07523da5-71ea-4253-9eb3-0cf8f954cd81',
  '3b2c061f-eec1-47df-8631-890d92a24ee9',
  'df5b5abb-71b4-4b58-ad3f-

Indexing completed
___

___
Retrieval process. Using vectorstore with similarity search here

In [39]:
retriever = chroma_store.as_retriever(
    search_type='similarity',
    search_kwargs={'k':10}
) 

In [40]:
from langchain_core.prompts import PromptTemplate


Retrieval is working, although the retrieved documents are not always good. Maybe that's just the side effect of using smaller local models
### Turns out using mean pooling somehow made things significantly better than the default "cls" pooling. This was true for both Qwen3-embedding as well as bge-m3

___

___
Augmentation

In [41]:
prompt_template = PromptTemplate(
    template='''
    You are a helpful assistant whose job is to answer questions based on the provided transcript context. Only answer based on the context given. If the context is insufficient, just say you don't know the answers to the questions.

    Context:
    {context}

    Question: {question}
    ''',
    input_variables=['context','question'],
    validate_template=True
)

In [42]:
# Tried the qeury "Does the video talk about The Great War at any point?"
docs = retriever.invoke("The Great War")

for i, doc in enumerate(docs):
    print(f"\n--- {i+1} ---")
    print(doc.page_content)


--- 1 ---
It's been a pleasure to hop on the microphone with Scott again and just talk about Elder Scrolls law >> and we will see you next time with a deep dive on one topic. >> No doubt again next time soon. I don't know if that's worn out or not. >> I [laughter] don't know.

--- 2 ---
The biggest thing though is like true seriously it's like I I like been on a bit of a like Warhammer 40k kick recently but it's just like and you you get you know get back in touch with the community seeing all the memes and stuff and it's just like the memes just get created by the community because people meme certain parts like but because it doesn't consciously make these it's a super serious sort of it's a heading that takes itself seriously. Just let that happen. Just take it serious and then >> Yeah. Yeah. Don't the memes will create themselves >> 100%. That's my lesson for anyone listening at Bethesda who might want to take some advice that's unrelated to law. The memes make themselves. >> We h

### I am still not able to pinpoint the exact problem to the local embedding model or not, so I will try using the exact same model using HF endpoint, and also a bigger model using HF endpoint, and see if the problem was with the model, local hosting problem or something else

In [43]:
from dotenv import load_dotenv
load_dotenv()

from langchain_huggingface import HuggingFaceEndpointEmbeddings

hf_qwen3_embedder = HuggingFaceEndpointEmbeddings(
    model='BAAI/bge-m3',
    provider='hf-inference',
    task='feature-extraction'
)

In [45]:
hf_vectorstore = Chroma.from_documents(
    embedding=hf_qwen3_embedder,
    documents=chunks,
    collection_name='hf_api_collection'
)

In [46]:
hf_retriever = hf_vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k':10
    }
)

In [47]:
hf_retrieved_docs = hf_retriever.invoke('Does the video talk about the Great war at any point?')

for i, doc in enumerate(hf_retrieved_docs):
    print(f"\n--- {i+1} ---")
    print(doc.page_content)


--- 1 ---
I think there's one of them that sounds like it's off the back of a it almost like on the back of Oblivion. That's the vibe. I can see the font, you know, the cover of the game. uh where it talks about the next coming war. >> Yeah, I can um I can find it here, but it's in the in the in the Great War book. So, it's like right at the end. There's literally a section called epilogue Hammerfell fights on alone, and it goes, you know, well, maybe we should should we explain the Great War first, a little bit of the context first. All right. So, we'll start us off with >> So, if it's Let's just assume it's Hammerfell. This is the Elder Scrolls 6. It's going to be Hammerfell podcast. We we can touch on some of the other things, but we're just going to >> small plug real quick. Upcoming episodes

--- 2 ---
but anyway so we should all be pretty familiar with the great war so it starts off towards the end of the fourth era 171 and it's basically the old Mary Dominion show up to Siridil

### So somehow, even for the exact same model, the HF Api is performing significantly better, while my locally hosted llama-server model is significantly worse. I have some work to do on why this could be the case. Don't think running the model using ollama as opposed to llama-server would be any better, given ollama is just a wrapper on top of llama-server in the first place, but man Im hella confused

___
Plan: <br>
1. Run the same local gguf using ollama instead of llama server and see performance <br>
2. If no performance improvement, then telling ollama to download the same model locally and see if perfomance improves

In [48]:
ollama_embedder = OpenAIEmbeddings(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    model="local-bge-m3",
    check_embedding_ctx_length=False
)

In [54]:
ollama_vectorstore.delete_collection()

ollama_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=ollama_embedder,
    collection_name='ollama_collection'
)

In [55]:
ollama_retriever = ollama_vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k':10
    }
)

In [56]:
ollama_retrieved_docs = ollama_retriever.invoke('Does the video talk about the Great war at any point?')

for i, doc in enumerate(ollama_retrieved_docs):
    print(f"\n--- {i+1} ---")
    print(doc.page_content)


--- 1 ---
I think there's one of them that sounds like it's off the back of a it almost like on the back of Oblivion. That's the vibe. I can see the font, you know, the cover of the game. uh where it talks about the next coming war. >> Yeah, I can um I can find it here, but it's in the in the in the Great War book. So, it's like right at the end. There's literally a section called epilogue Hammerfell fights on alone, and it goes, you know, well, maybe we should should we explain the Great War first, a little bit of the context first. All right. So, we'll start us off with >> So, if it's Let's just assume it's Hammerfell. This is the Elder Scrolls 6. It's going to be Hammerfell podcast. We we can touch on some of the other things, but we're just going to >> small plug real quick. Upcoming episodes

--- 2 ---
but anyway so we should all be pretty familiar with the great war so it starts off towards the end of the fourth era 171 and it's basically the old Mary Dominion show up to Siridil

Ok. At least running through ollama is still sucking haha. Now I am really considering that the gguf model that I downloaded might be ass or something

Will try using ollama's official bge-m3 and see if things are better or not. If yes, that will almost definitely confirm that the problem was with the gguf and not launching through llama-server

In [63]:
ollama_official_vectorstore.delete_collection()

ollama_official_embedder = OpenAIEmbeddings(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    model="bge-m3",
    check_embedding_ctx_length=False,
)


ollama_official_vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=ollama_official_embedder,
    collection_name='ollama_collection'
)

In [64]:
ollama_official_retriever = ollama_official_vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k':10
    }
)

In [65]:
ollama_official_retrieved_docs = ollama_official_retriever.invoke('Does the video talk about the Great war at any point?')

for i, doc in enumerate(ollama_official_retrieved_docs):
    print(f"\n--- {i+1} ---")
    print(doc.page_content)


--- 1 ---
I think there's one of them that sounds like it's off the back of a it almost like on the back of Oblivion. That's the vibe. I can see the font, you know, the cover of the game. uh where it talks about the next coming war. >> Yeah, I can um I can find it here, but it's in the in the in the Great War book. So, it's like right at the end. There's literally a section called epilogue Hammerfell fights on alone, and it goes, you know, well, maybe we should should we explain the Great War first, a little bit of the context first. All right. So, we'll start us off with >> So, if it's Let's just assume it's Hammerfell. This is the Elder Scrolls 6. It's going to be Hammerfell podcast. We we can touch on some of the other things, but we're just going to >> small plug real quick. Upcoming episodes

--- 2 ---
but anyway so we should all be pretty familiar with the great war so it starts off towards the end of the fourth era 171 and it's basically the old Mary Dominion show up to Siridil

Holy, that almost confirms that the gguf I downloaded was the issue, because ollama's official model is doing great

In [66]:
query = "Does the video talk about The Great War?"

hf_vec = hf_qwen3_embedder.embed_query(query)
local_vec = ollama_embedder.embed_query(query)
official_vec = ollama_official_embedder.embed_query(query)

print(len(hf_vec))
print(len(local_vec))
print(len(official_vec))

1024
1024
1024


OH MY GOD, I think I was using the 8bit quantized version, compared to 16bit from the official version. That explains goddamn everything holy. I'm sticking to llama-server now then, not using ollama, and if the performance of personally newly downloaded gguf is also trash, ill give up all hope on llama-server method and just use ollama